In [3]:
import pandas as pd
import plotly.express as px
import folium
import webbrowser
import os


## Plotting the stations 

## Plotting the stations with connections

#### Making dataset for the plot

In [26]:
file_path = "publicdataexportv131450706334_with_lon_lat.xlsx"

# -----------------------
# 1. BUS -> NODES
# -----------------------
bus = pd.read_excel(file_path, sheet_name="Bus", header=3)
bus.columns = [str(c).strip() for c in bus.columns]

# Clean numeric columns
bus["Bus Index"] = pd.to_numeric(bus["Bus Index"], errors="coerce")
bus["lon"] = pd.to_numeric(bus["lon"], errors="coerce")
bus["lat"] = pd.to_numeric(bus["lat"], errors="coerce")

bus["Voltage base[kV]"] = (
    bus["Voltage base[kV]"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)
bus["Voltage base[kV]"] = pd.to_numeric(bus["Voltage base[kV]"], errors="coerce")

nodes = bus[[
    "Bus Index",
    "Bus Name",
    "Station Full Name",
    "Location Name",
    "Voltage base[kV]",
    "lon",
    "lat"
]].copy()

nodes = nodes.rename(columns={
    "Bus Index": "bus_index",
    "Bus Name": "bus_name",
    "Station Full Name": "station_full_name",
    "Location Name": "area_code",
    "Voltage base[kV]": "voltage_kv"
})

nodes = nodes.dropna(subset=["bus_index"]).drop_duplicates(subset=["bus_index"])
nodes["bus_index"] = nodes["bus_index"].astype(int)

# -----------------------
# 2. LINE -> EDGES
# -----------------------
line = pd.read_excel(file_path, sheet_name="Line", header=3)
line.columns = [str(c).strip() for c in line.columns]

# Show columns once so you can inspect available edge attributes
print("Line columns:")
print(line.columns.tolist())

line["Node 1"] = pd.to_numeric(line["Node 1"], errors="coerce")
line["Node 2"] = pd.to_numeric(line["Node 2"], errors="coerce")

edges = line.copy()
edges = edges.rename(columns={
    "Node 1": "node1",
    "Node 2": "node2"
})

edges = edges.dropna(subset=["node1", "node2"]).copy()
edges["node1"] = edges["node1"].astype(int)
edges["node2"] = edges["node2"].astype(int)

# Optional: add coordinates and station names for plotting
node_lookup = nodes[[
    "bus_index", "station_full_name", "lat", "lon", "voltage_kv"
]].copy()

edges = edges.merge(
    node_lookup.add_prefix("n1_"),
    left_on="node1",
    right_on="n1_bus_index",
    how="left"
)

edges = edges.merge(
    node_lookup.add_prefix("n2_"),
    left_on="node2",
    right_on="n2_bus_index",
    how="left"
)

# Add edge_id
edges.insert(0, "edge_id", range(1, len(edges) + 1))

print("Nodes shape:", nodes.shape)
print("Edges shape:", edges.shape)

# -----------------------
# 3. SAVE CLEAN DATASET
# -----------------------
out_file = "danish_grid_cleaned.xlsx"
with pd.ExcelWriter(out_file, engine="openpyxl") as writer:
    nodes.to_excel(writer, sheet_name="nodes", index=False)
    edges.to_excel(writer, sheet_name="edges_lines", index=False)

print(f"Saved cleaned dataset to {out_file}")

Line columns:
['Node 1', 'Node 2', 'Line name', 'Area Name', 'Line type', 'Nominal Current[kA]', 'Nominal Voltage[kV]', 'R1[Ohm]', 'X1[Ohm]', 'G1[uS]', 'B1[uS]', 'Length[km]']
Nodes shape: (300, 7)
Edges shape: (358, 23)
Saved cleaned dataset to danish_grid_cleaned.xlsx


## Plotting 

In [27]:
file_path = "danish_grid_cleaned.xlsx"

nodes = pd.read_excel(file_path, sheet_name="nodes")
edges = pd.read_excel(file_path, sheet_name="edges_lines")

nodes["lat"] = pd.to_numeric(nodes["lat"], errors="coerce")
nodes["lon"] = pd.to_numeric(nodes["lon"], errors="coerce")
nodes["voltage_kv"] = pd.to_numeric(nodes["voltage_kv"], errors="coerce")

plot_nodes = nodes.dropna(subset=["lat", "lon"]).copy()

def get_color(v):
    if pd.isna(v):
        return "gray"
    if v >= 350:
        return "red"
    elif v >= 200:
        return "orange"
    elif v >= 100:
        return "blue"
    else:
        return "green"

m = folium.Map(
    location=[56.2, 10.0],
    zoom_start=7,
    tiles="CartoDB Positron"
)

# edges
for _, row in edges.iterrows():
    lat1 = row.get("n1_lat")
    lon1 = row.get("n1_lon")
    lat2 = row.get("n2_lat")
    lon2 = row.get("n2_lon")

    if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
        continue

    folium.PolyLine(
        locations=[(lat1, lon1), (lat2, lon2)],
        weight=2,
        opacity=0.5
    ).add_to(m)

# nodes
for _, row in plot_nodes.iterrows():
    color = get_color(row["voltage_kv"])

    popup_text = f"""
    <b>{row['station_full_name']}</b><br>
    Voltage: {row['voltage_kv']} kV<br>
    Area: {row['area_code']}
    """

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(popup_text, max_width=300)
    ).add_to(m)

out_file = "danish_grid_network_colored.html"
m.save(out_file)
webbrowser.open("file://" + os.path.realpath(out_file))

print("Map saved as:", out_file)

Map saved as: danish_grid_network_colored.html
